In [5]:
import torch
import torchaudio
import os

# 1. Ensure the directory exists
os.makedirs("./data", exist_ok=True)

# 2. Load a sample from the YesNo dataset
dataset = torchaudio.datasets.YESNO(root="./data", download=True)
audio, sr, _ = dataset[0]  # Get the first sample

# 3. Resample to 22,050 Hz
if sr != 22050:
    resample = torchaudio.transforms.Resample(sr, 22050)
    audio = resample(audio)

# 4. Convert to mono if stereo
if audio.shape[0] > 1:
    audio = torch.mean(audio, dim=0, keepdim=True)

# 5. Trim/pad to 59,049 samples
target_length = 59049
current_length = audio.shape[1]

if current_length > target_length:
    audio = audio[:, :target_length]
elif current_length < target_length:
    padding = target_length - current_length
    audio = torch.nn.functional.pad(audio, (0, padding))

# 6. Add batch dimension: (1, 1, 59049)
audio = audio.unsqueeze(0)

print(f"Final shape: {audio.shape}") # Should be torch.Size([1, 1, 59049])


Final shape: torch.Size([1, 1, 59049])


In [6]:
!pip install onnxruntime

In [8]:
import urllib.request
import onnxruntime as ort

# Direct URL to the ONNX model in the repo
url = "https://github.com/Spijkervet/CLMR/raw/master/examples/clmr-onnxruntime-web/clmr_sample-cnn.onnx"

# Download the ONNX file locally
model_path = "clmr_sample-cnn.onnx"
urllib.request.urlretrieve(url, model_path)

# Load the ONNX model with onnxruntime
session = ort.InferenceSession(model_path)

print("ONNX model loaded successfully!")

ONNX model loaded successfully!


In [9]:
outputs = session.run(None, {"audio": audio.numpy()})
representation = outputs[0]  # Shape: (1, 512)

In [10]:
representation

array([[-2.9238067, -1.6257615, -2.3130596, -6.550149 , -3.2325213,
        -5.950371 , -6.218897 , -6.066141 , -5.25144  , -1.0061903,
        -5.9188604, -5.6412983, -4.4499927, -3.6398435, -6.2137704,
        -4.1124334, -7.8190594, -3.995522 , -4.2679157, -4.4372106,
        -4.5234065, -4.203422 , -5.2963552, -7.1905956, -2.4396024,
        -4.2654643, -4.7696657, -5.080904 , -4.185429 , -6.9193006,
        -3.136317 , -8.10634  , -3.3902078, -4.9980106, -3.9678984,
        -4.9216003, -4.2390013, -5.97608  , -7.6658297, -4.5803676,
        -4.6812935, -6.5180674, -4.973977 , -5.609955 , -4.7118797,
        -6.3572335, -7.4903316, -6.4188423, -5.61098  , -4.9765143]],
      dtype=float32)